# Stock Market Data Analyzer - Analysis Example

This notebook demonstrates how to use the Stock Market Data Analyzer for comprehensive stock analysis.

## 📋 Table of Contents
1. [Setup and Imports](#setup)
2. [Data Fetching](#data-fetching)
3. [Data Cleaning](#data-cleaning)
4. [Technical Indicators](#technical-indicators)
5. [Trading Signals](#trading-signals)
6. [Backtesting](#backtesting)
7. [Visualization](#visualization)
8. [Report Generation](#report-generation)

## 🔧 Setup and Imports

Import the necessary modules and set up the analysis environment.

In [ ]:
# Import necessary modules
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Add src directory to path
sys.path.append('../src')

# Import our custom modules
from data_fetcher import StockDataFetcher
from data_cleaner import StockDataCleaner
from technical_indicators import TechnicalIndicators
from backtest_engine import BacktestEngine
from visualizer import StockVisualizer
from report_generator import ReportGenerator

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All modules imported successfully!")

## 📊 Data Fetching

Fetch historical stock data for analysis.

In [ ]:
# Initialize data fetcher
fetcher = StockDataFetcher()

# Define analysis parameters
symbol = "AAPL"
start_date = "2023-01-01"
end_date = "2023-12-31"

print(f"📈 Fetching data for {symbol} from {start_date} to {end_date}")

# Fetch stock data
raw_data = fetcher.fetch_data(symbol, start_date, end_date)

if not raw_data.empty:
    print(f"✅ Successfully fetched {len(raw_data)} days of data")
    print(f"📊 Data shape: {raw_data.shape}")
    print(f"💰 Price range: ${raw_data['close'].min():.2f} - ${raw_data['close'].max():.2f}")
    display(raw_data.head())
else:
    print("❌ No data fetched!")

## 🧹 Data Cleaning

Clean and validate the fetched data to ensure quality.

In [ ]:
# Initialize data cleaner
cleaner = StockDataCleaner()

print("🧹 Cleaning data...")

# Clean the data
clean_data, cleaning_report = cleaner.clean_data(raw_data, symbol)

print(f"✅ Data cleaning completed")
print(f"📊 Data quality score: {cleaning_report['data_quality_score']:.2f}/100")
print(f"📉 Original shape: {cleaning_report['original_shape']}")
print(f"📈 Final shape: {cleaning_report['final_shape']}")

# Display cleaning summary
print("\n🔧 Cleaning Steps:")
for step in cleaning_report['cleaning_steps']:
    print(f"  - {step['step']}: {step.get('removed_count', 0)} items processed")

display(clean_data.head())

## 📈 Technical Indicators

Calculate comprehensive technical indicators for analysis.

In [ ]:
# Initialize technical indicators calculator
indicators = TechnicalIndicators()

print("📈 Calculating technical indicators...")

# Calculate all indicators
indicators_data = indicators.calculate_all_indicators(clean_data)

print(f"✅ Technical indicators calculated")
print(f"📊 New columns added: {len(indicators_data.columns) - len(clean_data.columns)}")
print(f"📈 Total columns: {len(indicators_data.columns)}")

# Display new indicator columns
indicator_columns = [col for col in indicators_data.columns if col not in clean_data.columns]
print(f"\n📊 New Indicators: {indicator_columns}")

# Show latest indicator values
latest_data = indicators_data.iloc[-1]
print("\n🔍 Latest Indicator Values:")
for col in ['sma_20', 'sma_50', 'rsi_14', 'macd', 'bb_upper', 'bb_lower']:
    if col in latest_data:
        print(f"  {col}: {latest_data[col]:.4f}")

display(indicators_data.tail())

## 🎯 Trading Signals

Generate trading signals based on technical indicators.

In [ ]:
# Generate trading signals using SMA crossover strategy
print("🎯 Generating trading signals (SMA Crossover)...")

signals_data = indicators.generate_trading_signals(indicators_data, 'sma_crossover')

# Analyze signals
signal_counts = signals_data['signal'].value_counts()
print(f"✅ Signals generated")
print(f"📊 Signal distribution:")
for signal, count in signal_counts.items():
    signal_name = {1: 'BUY', -1: 'SELL', 0: 'HOLD'}.get(signal, 'UNKNOWN')
    print(f"  {signal_name}: {count} occurrences")

# Find recent signals
recent_signals = signals_data[signals_data['signal'] != 0].tail(10)
if not recent_signals.empty:
    print("\n📅 Recent Trading Signals:")
    for date, row in recent_signals.iterrows():
        signal_name = {1: 'BUY', -1: 'SELL'}.get(row['signal'], 'UNKNOWN')
        print(f"  {date.strftime('%Y-%m-%d')}: {signal_name} at ${row['close']:.2f}")

display(signals_data[['close', 'sma_20', 'sma_50', 'signal']].tail(10))

## 💰 Backtesting

Backtest the trading strategy to evaluate performance.

In [ ]:
# Initialize backtest engine
backtest = BacktestEngine(initial_capital=100000)

print("💰 Running backtest...")

# Run backtest on the signals
backtest_results = backtest.run_backtest(signals_data, 'signal')

if 'error' not in backtest_results:
    print("✅ Backtest completed successfully")
    print(f"💰 Initial Capital: ${backtest_results['initial_capital']:,.2f}")
    print(f"💵 Final Value: ${backtest_results['final_value']:,.2f}")
    print(f"📈 Total Return: {backtest_results['total_return']:.2%}")
    print(f"📊 Annualized Return: {backtest_results.get('annualized_return', 0):.2%}")
    print(f"⚡ Sharpe Ratio: {backtest_results['sharpe_ratio']:.2f}")
    print(f"📉 Max Drawdown: {backtest_results['max_drawdown']:.2%}")
    print(f"🎯 Win Rate: {backtest_results['win_rate']:.2%}")
    print(f"📊 Total Trades: {backtest_results['total_trades']}")
    print(f"💡 Profit Factor: {backtest_results.get('profit_factor', 0):.2f}")
else:
    print(f"❌ Backtest failed: {backtest_results['error']}")

## 📊 Visualization

Create comprehensive visualizations of the analysis results.

In [ ]:
# Initialize visualizer
visualizer = StockVisualizer()

# Create price chart with moving averages
print("📊 Creating price chart...")
price_chart = visualizer.create_price_chart(indicators_data, symbol)

# Create technical indicators dashboard
print("📈 Creating technical dashboard...")
dashboard = visualizer.create_summary_dashboard(indicators_data, symbol)

# Create returns analysis
if 'returns' in indicators_data.columns:
    print("📉 Creating returns analysis...")
    returns_chart = visualizer.create_returns_analysis(indicators_data, symbol)

# Create backtest visualization
if 'equity_curve' in backtest_results and 'trades' in backtest_results:
    print("💰 Creating backtest chart...")
    backtest_chart = visualizer.create_backtest_chart(
        backtest_results['equity_curve'], 
        backtest_results['trades'], 
        symbol
    )

print("✅ All visualizations created successfully!")
print("📁 Check the outputs/ folder for generated charts")

## 📄 Report Generation

Generate a comprehensive analysis report.

In [ ]:
# Initialize report generator
reporter = ReportGenerator()

print("📄 Generating comprehensive report...")

# Get indicators summary
indicators_summary = indicators.get_indicator_summary(indicators_data)

# Generate comprehensive report
report_path = reporter.generate_comprehensive_report(
    indicators_data, 
    symbol, 
    backtest_results, 
    indicators_summary
)

if report_path:
    print(f"✅ Report generated successfully!")
    print(f"📁 Report saved to: {report_path}")
else:
    print("❌ Report generation failed")

# Generate summary report
summary_data = {
    'symbol': symbol,
    'basic_stats': reporter._calculate_basic_statistics(indicators_data),
    'risk_metrics': reporter._calculate_risk_metrics(indicators_data),
    'performance_metrics': backtest_results,
    'recommendations': reporter._generate_recommendations(indicators_data, backtest_results)
}

summary = reporter.generate_summary_report(summary_data)
print("\n📋 Analysis Summary:")
print(summary)

## 🎯 Custom Analysis

Try your own analysis with different parameters.

In [ ]:
# Try different symbols
symbols = ['GOOGL', 'MSFT', 'TSLA']
results = {}

for sym in symbols:
    print(f"\n📈 Analyzing {sym}...")
    
    # Fetch data
    data = fetcher.fetch_data(sym, start_date, end_date)
    if data.empty:
        print(f"❌ No data for {sym}")
        continue
    
    # Clean and calculate indicators
    clean_data_sym, _ = cleaner.clean_data(data, sym)
    indicators_data_sym = indicators.calculate_all_indicators(clean_data_sym)
    signals_data_sym = indicators.generate_trading_signals(indicators_data_sym, 'sma_crossover')
    
    # Run backtest
    backtest_results_sym = backtest.run_backtest(signals_data_sym, 'signal')
    
    if 'error' not in backtest_results_sym:
        results[sym] = {
            'return': backtest_results_sym['total_return'],
            'sharpe': backtest_results_sym['sharpe_ratio'],
            'max_dd': backtest_results_sym['max_drawdown'],
            'win_rate': backtest_results_sym['win_rate']
        }
        print(f"  ✅ {sym}: {backtest_results_sym['total_return']:.2%} return, {backtest_results_sym['sharpe_ratio']:.2f} Sharpe")
    else:
        print(f"  ❌ {sym}: Backtest failed")

# Create comparison table
comparison_df = pd.DataFrame(results).T
print("\n📊 Strategy Comparison:")
display(comparison_df)

# Find best performing stock
best_stock = comparison_df['return'].idxmax()
print(f"\n🏆 Best performing stock: {best_stock} with {comparison_df.loc[best_stock, 'return']:.2%} return")

## 📚 Key Takeaways

### What We Learned:
1. **Data Quality**: Clean data is essential for accurate analysis
2. **Technical Indicators**: Multiple indicators provide better signals
3. **Backtesting**: Realistic assumptions are crucial for strategy evaluation
4. **Risk Management**: Returns must be evaluated relative to risk
5. **Visualization**: Charts help understand complex relationships

### Next Steps:
- Try different strategies (RSI, MACD, Bollinger Bands)
- Optimize strategy parameters
- Implement risk management rules
- Add fundamental analysis
- Create portfolio optimization

### 🎉 Congratulations!
You've successfully completed a comprehensive stock market analysis using the Stock Market Data Analyzer!